# 05 — Integrated Drowsiness Detection System

## Part 1 — Camera Management

This notebook will eventually combine:


Part 1 only initializes and manages the camera input.

The camera source must be interchangeable so the system can use:


The rest of the application must receive frames through the
camera manager and must not depend on a specific camera index.

In [27]:
# ============================================================
# NB5 — PART 1
# IMPORT LIBRARIES
# ============================================================

import cv2
import time

print("=" * 60)
print("INTEGRATED DROWSINESS SYSTEM")
print("PART 1 — CAMERA MANAGEMENT")
print("=" * 60)

print("OpenCV:", cv2.__version__)

INTEGRATED DROWSINESS SYSTEM
PART 1 — CAMERA MANAGEMENT
OpenCV: 5.0.0


In [28]:
# ============================================================
# CAMERA CONFIGURATION
# ============================================================

# Camera index.
#
# Typical examples:
# 0 = first/default camera
# 1 = second camera
#
# Change this value when selecting the USB webcam.
CAMERA_INDEX = 0

# Requested camera resolution
FRAME_WIDTH = 1280
FRAME_HEIGHT = 720

# Requested FPS
TARGET_FPS = 30

# Mirror the displayed camera image.
#
# True:
#   behaves like a normal selfie/mirror view
#
# False:
#   keeps the original camera orientation
MIRROR_CAMERA = True


print("=" * 60)
print("CAMERA CONFIGURATION")
print("=" * 60)

print("Camera Index :", CAMERA_INDEX)
print("Resolution   :", f"{FRAME_WIDTH} x {FRAME_HEIGHT}")
print("Target FPS   :", TARGET_FPS)
print("Mirror       :", MIRROR_CAMERA)

print("=" * 60)

CAMERA CONFIGURATION
Camera Index : 0
Resolution   : 1280 x 720
Target FPS   : 30
Mirror       : True


In [29]:
# ============================================================
# CAMERA DISCOVERY
# ============================================================

def find_available_cameras(max_cameras=10):
    """
    Check common OpenCV camera indices and return the
    indices that can successfully be opened.
    """

    available = []

    print("=" * 60)
    print("SEARCHING FOR AVAILABLE CAMERAS")
    print("=" * 60)

    for index in range(max_cameras):

        cap = cv2.VideoCapture(index)

        if cap.isOpened():

            available.append(index)

            print(
                f"Camera {index}: AVAILABLE"
            )

            cap.release()

        else:

            print(
                f"Camera {index}: NOT AVAILABLE"
            )

            cap.release()

    print("=" * 60)

    print(
        "Available camera indices:",
        available
    )

    print("=" * 60)

    return available


available_cameras = find_available_cameras()

SEARCHING FOR AVAILABLE CAMERAS
Camera 0: AVAILABLE
Camera 1: NOT AVAILABLE
Camera 2: NOT AVAILABLE
Camera 3: NOT AVAILABLE
Camera 4: NOT AVAILABLE
Camera 5: NOT AVAILABLE
Camera 6: NOT AVAILABLE
Camera 7: NOT AVAILABLE
Camera 8: NOT AVAILABLE
Camera 9: NOT AVAILABLE
Available camera indices: [0]


In [30]:
# ============================================================
# CAMERA MANAGER
# ============================================================

class CameraManager:

    def __init__(
        self,
        camera_index=0,
        width=1280,
        height=720,
        fps=30,
        mirror=True
    ):

        self.camera_index = camera_index

        self.width = width
        self.height = height
        self.fps = fps

        self.mirror = mirror

        self.cap = None

        self.is_open = False


    def open(self):

        """
        Open the selected camera and apply the requested
        resolution and FPS settings.
        """

        if self.is_open:

            return True


        self.cap = cv2.VideoCapture(
            self.camera_index
        )


        if not self.cap.isOpened():

            self.cap.release()

            self.cap = None

            self.is_open = False

            return False


        # ----------------------------------------------------
        # Requested camera settings
        # ----------------------------------------------------

        self.cap.set(
            cv2.CAP_PROP_FRAME_WIDTH,
            self.width
        )

        self.cap.set(
            cv2.CAP_PROP_FRAME_HEIGHT,
            self.height
        )

        self.cap.set(
            cv2.CAP_PROP_FPS,
            self.fps
        )


        self.is_open = True

        return True


    def read(self):

        """
        Read one frame from the selected camera.

        Returns:
            ret, frame
        """

        if not self.is_open:

            raise RuntimeError(
                "Camera is not open."
            )


        ret, frame = self.cap.read()


        if not ret:

            return False, None


        # ----------------------------------------------------
        # Mirror frame if requested
        # ----------------------------------------------------

        if self.mirror:

            frame = cv2.flip(
                frame,
                1
            )


        return True, frame


    def get_actual_settings(self):

        """
        Return the camera settings actually reported by
        OpenCV after opening the camera.
        """

        if not self.is_open:

            raise RuntimeError(
                "Camera is not open."
            )


        actual_width = int(
            self.cap.get(
                cv2.CAP_PROP_FRAME_WIDTH
            )
        )

        actual_height = int(
            self.cap.get(
                cv2.CAP_PROP_FRAME_HEIGHT
            )
        )

        actual_fps = self.cap.get(
            cv2.CAP_PROP_FPS
        )


        return {
            "width": actual_width,
            "height": actual_height,
            "fps": actual_fps
        }


    def release(self):

        """
        Release the camera.
        """

        if self.cap is not None:

            self.cap.release()


        self.cap = None

        self.is_open = False


    def __enter__(self):

        if not self.open():

            raise RuntimeError(
                f"Unable to open camera "
                f"index {self.camera_index}"
            )

        return self


    def __exit__(
        self,
        exc_type,
        exc_value,
        traceback
    ):

        self.release()


print("CameraManager class created successfully.")

CameraManager class created successfully.


In [31]:
# ============================================================
# CREATE CAMERA MANAGER
# ============================================================

camera = CameraManager(
    camera_index=CAMERA_INDEX,
    width=FRAME_WIDTH,
    height=FRAME_HEIGHT,
    fps=TARGET_FPS,
    mirror=MIRROR_CAMERA
)

print("=" * 60)
print("CAMERA MANAGER CREATED")
print("=" * 60)

print("Selected Camera :", CAMERA_INDEX)
print("Mirror          :", MIRROR_CAMERA)

print("=" * 60)

CAMERA MANAGER CREATED
Selected Camera : 0
Mirror          : True


In [32]:
# ============================================================
# OPEN AND VERIFY CAMERA
# ============================================================

if not camera.open():

    raise RuntimeError(
        f"Could not open camera index "
        f"{CAMERA_INDEX}"
    )


actual_settings = (
    camera.get_actual_settings()
)


print("=" * 60)
print("CAMERA OPENED SUCCESSFULLY")
print("=" * 60)

print(
    "Requested Resolution :",
    f"{FRAME_WIDTH} x {FRAME_HEIGHT}"
)

print(
    "Actual Resolution    :",
    f"{actual_settings['width']} "
    f"x "
    f"{actual_settings['height']}"
)

print(
    "Requested FPS        :",
    TARGET_FPS
)

print(
    "Actual FPS           :",
    f"{actual_settings['fps']:.2f}"
)

print(
    "Mirror               :",
    MIRROR_CAMERA
)

print("=" * 60)

CAMERA OPENED SUCCESSFULLY
Requested Resolution : 1280 x 720
Actual Resolution    : 1280 x 720
Requested FPS        : 30
Actual FPS           : 30.00
Mirror               : True


In [33]:
# ============================================================
# CAMERA PREVIEW TEST
# ============================================================

print("=" * 60)
print("CAMERA PREVIEW")
print("=" * 60)
print("Press Q to stop the preview.")
print("=" * 60)


while True:

    ret, frame = camera.read()


    if not ret:

        print(
            "Failed to read frame from camera."
        )

        break


    # --------------------------------------------------------
    # Display frame
    # --------------------------------------------------------

    cv2.imshow(
        "NB5 - Camera Preview",
        frame
    )


    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


cv2.destroyAllWindows()

camera.release()

print("Camera preview stopped.")

CAMERA PREVIEW
Press Q to stop the preview.
Camera preview stopped.


## Camera Selection

To use another camera:

1. Run the camera discovery cell.
2. Check the available camera indices.
3. Change `CAMERA_INDEX`.
4. Recreate the `CameraManager`.
5. Run the camera verification cell.

Example:

```python
CAMERA_INDEX = 1
```

In [34]:
# ============================================================
# NB5 — PART 2
# IMPORT MEDIAPIPE + PYTORCH
# ============================================================

import mediapipe as mp

from mediapipe.tasks import python
from mediapipe.tasks.python import vision

import torch
import torch.nn as nn

import numpy as np

print("=" * 60)
print("PART 2 — MODEL / LANDMARK INITIALIZATION")
print("=" * 60)

print("MediaPipe :", mp.__version__)
print("PyTorch   :", torch.__version__)
print("NumPy     :", np.__version__)

PART 2 — MODEL / LANDMARK INITIALIZATION
MediaPipe : 1.0.0
PyTorch   : 2.11.0+cu128
NumPy     : 2.2.6


In [35]:
# ============================================================
# DEVICE + PROJECT PATHS
# ============================================================
from pathlib import Path

PROJECT_ROOT = Path.cwd()

MEDIAPIPE_MODEL_PATH = (
    PROJECT_ROOT /
    "assets" /
    "face_landmarker.task"
)

MOUTH_MODEL_PATH = (
    PROJECT_ROOT /
    "models" /
    "mouth_cnn_best.pth"
)

device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("=" * 60)
print("DEVICE + MODEL PATHS")
print("=" * 60)

print("Project Root :", PROJECT_ROOT)
print("MediaPipe Model :", MEDIAPIPE_MODEL_PATH)
print("Mouth Model :", MOUTH_MODEL_PATH)
print("Device :", device)

print("=" * 60)

if not MEDIAPIPE_MODEL_PATH.exists():

    raise FileNotFoundError(
        f"MediaPipe model not found:\n"
        f"{MEDIAPIPE_MODEL_PATH}"
    )

if not MOUTH_MODEL_PATH.exists():

    raise FileNotFoundError(
        f"Mouth CNN model not found:\n"
        f"{MOUTH_MODEL_PATH}"
    )

DEVICE + MODEL PATHS
Project Root : c:\Users\Omswaroop\OneDrive\Desktop\New folder
MediaPipe Model : c:\Users\Omswaroop\OneDrive\Desktop\New folder\assets\face_landmarker.task
Mouth Model : c:\Users\Omswaroop\OneDrive\Desktop\New folder\models\mouth_cnn_best.pth
Device : cuda


In [36]:
# ============================================================
# INITIALIZE MEDIAPIPE FACE LANDMARKER
# ============================================================
BaseOptions = python.BaseOptions

FaceLandmarker = vision.FaceLandmarker

FaceLandmarkerOptions = (
    vision.FaceLandmarkerOptions
)

RunningMode = vision.RunningMode

landmarker_options = FaceLandmarkerOptions(

    base_options=BaseOptions(
        model_asset_path=str(
            MEDIAPIPE_MODEL_PATH
        )
    ),

    running_mode=RunningMode.IMAGE,

    num_faces=1
)

face_landmarker = (
    FaceLandmarker.create_from_options(
        landmarker_options
    )
)

print("=" * 60)
print("MEDIAPIPE FACE LANDMARKER READY")
print("=" * 60)

print("Model :", MEDIAPIPE_MODEL_PATH)
print("Mode : IMAGE")
print("Faces : 1")

print("=" * 60)

MEDIAPIPE FACE LANDMARKER READY
Model : c:\Users\Omswaroop\OneDrive\Desktop\New folder\assets\face_landmarker.task
Mode : IMAGE
Faces : 1


In [37]:
# ============================================================
# MOUTH CNN ARCHITECTURE
# ============================================================
NUM_CLASSES = 3

CLASS_NAMES = [
    "Closed",
    "Talking",
    "Yawn"
]

class CustomCNN(nn.Module):

    def __init__(self):

        super().__init__()

        self.features = nn.Sequential(

            nn.Conv2d(
                1,
                32,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(32),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                32,
                64,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(64),

            nn.ReLU(),

            nn.MaxPool2d(2),

            nn.Conv2d(
                64,
                128,
                kernel_size=3,
                padding=1
            ),

            nn.BatchNorm2d(128),

            nn.ReLU(),

            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(
                128 * 16 * 16,
                256
            ),

            nn.ReLU(),

            nn.Dropout(0.5),

            nn.Linear(
                256,
                NUM_CLASSES
            )
        )

    def forward(self, x):

        x = self.features(x)

        x = self.classifier(x)

        return x

print("Mouth CNN architecture defined.")

Mouth CNN architecture defined.


In [38]:
# ============================================================
# LOAD TRAINED MOUTH CNN
# ============================================================
mouth_model = CustomCNN().to(device)

mouth_model.load_state_dict(
    torch.load(
        MOUTH_MODEL_PATH,
        map_location=device
    )
)

mouth_model.eval()

print("=" * 60)
print("TRAINED MOUTH CNN LOADED")
print("=" * 60)

print("Model :", MOUTH_MODEL_PATH)
print("Device:", device)
print("Classes:", CLASS_NAMES)

print("=" * 60)

TRAINED MOUTH CNN LOADED
Model : c:\Users\Omswaroop\OneDrive\Desktop\New folder\models\mouth_cnn_best.pth
Device: cuda
Classes: ['Closed', 'Talking', 'Yawn']


In [39]:
# ============================================================
# MOUTH CNN PREPROCESSING
# ============================================================
from PIL import Image
from torchvision import transforms

mouth_transform = transforms.Compose([

    transforms.Grayscale(
        num_output_channels=1
    ),

    transforms.Resize(
        (128, 128)
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.5],
        std=[0.5]
    )
])

print("Mouth preprocessing pipeline ready.")

Mouth preprocessing pipeline ready.


In [40]:
# ============================================================
# MOUTH LANDMARK INDICES
# ============================================================
MOUTH_POINTS = [

    61, 146, 91, 181, 84,

    17, 314, 405, 321, 375,

    291, 409, 270, 269, 267,

    0, 37, 39, 40, 185
]

print(
f"Mouth landmarks configured: "
 f"{len(MOUTH_POINTS)} points"
)

Mouth landmarks configured: 20 points


In [41]:
# ============================================================
# EYE LANDMARK INDICES
# ============================================================
LEFT_EYE = [
    33,
    160,
    158,
    133,
    153,
    144
]

RIGHT_EYE = [
    362,
    385,
    387,
    263,
    373,
    380
]

print("Left eye landmarks :", LEFT_EYE)
print("Right eye landmarks:", RIGHT_EYE)

Left eye landmarks : [33, 160, 158, 133, 153, 144]
Right eye landmarks: [362, 385, 387, 263, 373, 380]


In [42]:
# ============================================================
# SHARED FRAME PROCESSING
# ============================================================
def detect_face_landmarks(frame):

    """
    Run MediaPipe Face Landmarker on one OpenCV frame.

    Returns:
        landmarks if a face is detected,
        otherwise None.
    """

    if frame is None:

        return None

    rgb = cv2.cvtColor(
        frame,
        cv2.COLOR_BGR2RGB
    )

    mp_image = mp.Image(
        image_format=mp.ImageFormat.SRGB,
        data=rgb
    )

    result = face_landmarker.detect(
        mp_image
    )

    if not result.face_landmarks:

        return None

    return result.face_landmarks[0]

print("Shared face-landmark function ready.")

Shared face-landmark function ready.


In [43]:
# ============================================================
# TEST CAMERA → MEDIAPIPE PIPELINE
# ============================================================

print("=" * 60)
print("TESTING SHARED FRAME PIPELINE")
print("=" * 60)
print("Press Q to stop.")
print("=" * 60)


# ------------------------------------------------------------
# Reopen camera if it was released by an earlier test
# ------------------------------------------------------------

if not camera.is_open:

    print("Camera is closed. Reopening...")

    if not camera.open():

        raise RuntimeError(
            f"Could not reopen camera index "
            f"{camera.camera_index}"
        )


while True:

    ret, frame = camera.read()


    if not ret:

        print(
            "Failed to read camera frame."
        )

        break


    # --------------------------------------------------------
    # Shared MediaPipe face landmarks
    # --------------------------------------------------------

    landmarks = detect_face_landmarks(
        frame
    )


    display_frame = frame.copy()


    if landmarks is not None:

        h, w = display_frame.shape[:2]


        # ----------------------------------------------------
        # Draw left eye landmarks
        # ----------------------------------------------------

        for idx in LEFT_EYE:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                3,
                (0, 255, 0),
                -1
            )


        # ----------------------------------------------------
        # Draw right eye landmarks
        # ----------------------------------------------------

        for idx in RIGHT_EYE:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                3,
                (0, 255, 0),
                -1
            )


        # ----------------------------------------------------
        # Draw mouth landmarks
        # ----------------------------------------------------

        for idx in MOUTH_POINTS:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                2,
                (255, 0, 0),
                -1
            )


        cv2.putText(
            display_frame,
            "Face landmarks detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 255, 0),
            2
        )


    else:

        cv2.putText(
            display_frame,
            "Face not detected",
            (20, 40),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.8,
            (0, 0, 255),
            2
        )


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    cv2.imshow(
        "NB5 - Shared Frame Pipeline",
        display_frame
    )


    # --------------------------------------------------------
    # Quit
    # --------------------------------------------------------

    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


cv2.destroyAllWindows()

# Release camera after test
camera.release()

print("=" * 60)
print("SHARED PIPELINE TEST COMPLETE")
print("=" * 60)

TESTING SHARED FRAME PIPELINE
Press Q to stop.
Camera is closed. Reopening...
SHARED PIPELINE TEST COMPLETE


In [44]:
# ============================================================
# PART 3 — EYE LANDMARK CONFIGURATION
# ============================================================

LEFT_EYE = [
    33,
    160,
    158,
    133,
    153,
    144
]


RIGHT_EYE = [
    362,
    385,
    387,
    263,
    373,
    380
]


print("=" * 60)
print("EYE LANDMARKS CONFIGURED")
print("=" * 60)

print("Left Eye :", LEFT_EYE)
print("Right Eye:", RIGHT_EYE)

print("=" * 60)

EYE LANDMARKS CONFIGURED
Left Eye : [33, 160, 158, 133, 153, 144]
Right Eye: [362, 385, 387, 263, 373, 380]


In [45]:
# ============================================================
# EAR CALCULATION
# ============================================================


def calculate_ear(
    landmarks,
    eye_indices,
    frame_width,
    frame_height
):

    """
    Calculate Eye Aspect Ratio (EAR)
    using six MediaPipe eye landmarks.
    """

    points = []

    for idx in eye_indices:

        x = (
            landmarks[idx].x
            * frame_width
        )

        y = (
            landmarks[idx].y
            * frame_height
        )

        points.append(
            np.array([x, y])
        )


    p1, p2, p3, p4, p5, p6 = points


    vertical_1 = np.linalg.norm(
        p2 - p6
    )


    vertical_2 = np.linalg.norm(
        p3 - p5
    )


    horizontal = np.linalg.norm(
        p1 - p4
    )


    if horizontal == 0:

        return 0.0


    ear = (
        vertical_1
        + vertical_2
    ) / (
        2.0 * horizontal
    )


    return float(ear)


print("EAR calculation function ready.")

EAR calculation function ready.


In [46]:
# ============================================================
# EYE CALIBRATION
# ============================================================

# Based on the calibration performed in 04_eye_analysis.ipynb:
#
# Eyes open  : approximately 0.32 – 0.38
# Eyes closed: approximately 0.025 – 0.061
#
# Initial threshold:
# midpoint between the observed open/closed ranges.

EAR_THRESHOLD = 0.19


def classify_eye_state(ear):

    """
    Classify eye as OPEN or CLOSED.
    """

    if ear >= EAR_THRESHOLD:

        return "OPEN"

    return "CLOSED"


print("=" * 60)
print("EYE STATE CLASSIFIER")
print("=" * 60)

print(
    f"EAR Threshold: {EAR_THRESHOLD:.3f}"
)

print(
    "EAR >= threshold -> OPEN"
)

print(
    "EAR < threshold  -> CLOSED"
)

print("=" * 60)

EYE STATE CLASSIFIER
EAR Threshold: 0.190
EAR >= threshold -> OPEN
EAR < threshold  -> CLOSED


In [47]:
# ============================================================
# EYE CLOSURE CONFIGURATION
# ============================================================

# Initial engineering thresholds.
#
# These are not medically validated values.
# They are used for the prototype system and will be tuned
# during later testing.

BLINK_THRESHOLD = 0.50

LONG_BLINK_THRESHOLD = 1.00

PROLONGED_CLOSURE_THRESHOLD = 1.00


print("=" * 60)
print("EYE CLOSURE CONFIGURATION")
print("=" * 60)

print(
    f"Blink threshold       : "
    f"{BLINK_THRESHOLD:.2f} sec"
)

print(
    f"Long blink threshold  : "
    f"{LONG_BLINK_THRESHOLD:.2f} sec"
)

print(
    f"Prolonged threshold   : "
    f"{PROLONGED_CLOSURE_THRESHOLD:.2f} sec"
)

print("=" * 60)

EYE CLOSURE CONFIGURATION
Blink threshold       : 0.50 sec
Long blink threshold  : 1.00 sec
Prolonged threshold   : 1.00 sec


In [48]:
# ============================================================
# EYE STATE ANALYSIS
# ============================================================


def analyze_eyes(
    landmarks,
    frame_width,
    frame_height
):

    """
    Calculate both-eye EAR and eye states
    from already-detected MediaPipe landmarks.

    MediaPipe is NOT called inside this function.
    """

    left_ear = calculate_ear(
        landmarks,
        LEFT_EYE,
        frame_width,
        frame_height
    )


    right_ear = calculate_ear(
        landmarks,
        RIGHT_EYE,
        frame_width,
        frame_height
    )


    average_ear = (
        left_ear
        + right_ear
    ) / 2.0


    left_state = classify_eye_state(
        left_ear
    )


    right_state = classify_eye_state(
        right_ear
    )


    both_closed = (
        left_state == "CLOSED"
        and
        right_state == "CLOSED"
    )


    if both_closed:

        combined_state = (
            "BOTH EYES CLOSED"
        )

    elif (
        left_state == "OPEN"
        and
        right_state == "OPEN"
    ):

        combined_state = (
            "BOTH EYES OPEN"
        )

    else:

        combined_state = (
            "ONE EYE CLOSED"
        )


    return {
        "left_ear": left_ear,
        "right_ear": right_ear,
        "average_ear": average_ear,
        "left_state": left_state,
        "right_state": right_state,
        "both_closed": both_closed,
        "combined_state": combined_state
    }


print("Eye analysis function ready.")

Eye analysis function ready.


In [49]:
# ============================================================
# EYE TEMPORAL STATE
# ============================================================

eye_temporal_state = {

    "eyes_closed": False,

    "closure_start_time": None,

    "current_closure_duration": 0.0,

    "last_completed_closure": 0.0,

    "last_closure_event": "None",

    "longest_closure": 0.0

}


print("=" * 60)
print("EYE TEMPORAL STATE INITIALIZED")
print("=" * 60)

print(eye_temporal_state)

print("=" * 60)

EYE TEMPORAL STATE INITIALIZED
{'eyes_closed': False, 'closure_start_time': None, 'current_closure_duration': 0.0, 'last_completed_closure': 0.0, 'last_closure_event': 'None', 'longest_closure': 0.0}


In [50]:
# ============================================================
# UPDATE EYE TEMPORAL STATE
# ============================================================


def update_eye_temporal_state(
    both_closed,
    current_time,
    state
):

    """
    Update eye closure timing based on the current
    both-eyes-closed state.

    Returns the updated state dictionary.
    """

    # --------------------------------------------------------
    # Eyes currently closed
    # --------------------------------------------------------

    if both_closed:

        if not state["eyes_closed"]:

            state["eyes_closed"] = True

            state["closure_start_time"] = (
                current_time
            )

            state["current_closure_duration"] = 0.0


        else:

            state["current_closure_duration"] = (
                current_time
                - state["closure_start_time"]
            )


    # --------------------------------------------------------
    # Eyes currently open
    # --------------------------------------------------------

    else:

        if state["eyes_closed"]:

            duration = (
                state["current_closure_duration"]
            )


            state["last_completed_closure"] = (
                duration
            )


            state["last_closure_event"] = (
                classify_closure_duration(
                    duration
                )
            )


            if duration > state["longest_closure"]:

                state["longest_closure"] = (
                    duration
                )


        state["eyes_closed"] = False

        state["closure_start_time"] = None

        state["current_closure_duration"] = 0.0


    return state

In [51]:
# ============================================================
# CLOSURE EVENT CLASSIFICATION
# ============================================================


def classify_closure_duration(
    duration
):

    if duration < BLINK_THRESHOLD:

        return "NORMAL BLINK"


    elif duration < LONG_BLINK_THRESHOLD:

        return "LONG BLINK"


    else:

        return "PROLONGED CLOSURE"


print("Closure event classification ready.")

Closure event classification ready.


In [52]:
# ============================================================
# PART 3 — LIVE EYE ANALYSIS TEST
# ============================================================

print("=" * 60)
print("LIVE EYE ANALYSIS TEST")
print("=" * 60)
print("Press Q to stop.")
print("=" * 60)


# ------------------------------------------------------------
# Reopen the shared camera if necessary
# ------------------------------------------------------------

if not camera.is_open:

    if not camera.open():

        raise RuntimeError(
            f"Could not open camera "
            f"index {camera.camera_index}"
        )


# ------------------------------------------------------------
# Reset temporal state
# ------------------------------------------------------------

eye_temporal_state = {

    "eyes_closed": False,

    "closure_start_time": None,

    "current_closure_duration": 0.0,

    "last_completed_closure": 0.0,

    "last_closure_event": "None",

    "longest_closure": 0.0
}


while True:

    ret, frame = camera.read()


    if not ret:

        print(
            "Failed to read camera frame."
        )

        break


    # --------------------------------------------------------
    # Shared MediaPipe call
    # --------------------------------------------------------

    landmarks = detect_face_landmarks(
        frame
    )


    display_frame = frame.copy()


    # --------------------------------------------------------
    # Face detected
    # --------------------------------------------------------

    if landmarks is not None:

        h, w = display_frame.shape[:2]


        # ----------------------------------------------------
        # Analyze eyes
        # ----------------------------------------------------

        eye_data = analyze_eyes(
            landmarks,
            w,
            h
        )


        # ----------------------------------------------------
        # Update temporal state
        # ----------------------------------------------------

        eye_temporal_state = (
            update_eye_temporal_state(
                eye_data["both_closed"],
                time.time(),
                eye_temporal_state
            )
        )


        # ----------------------------------------------------
        # Display EAR
        # ----------------------------------------------------

        cv2.putText(
            display_frame,
            f"Left EAR: "
            f"{eye_data['left_ear']:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Right EAR: "
            f"{eye_data['right_ear']:.3f}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Average EAR: "
            f"{eye_data['average_ear']:.3f}",
            (20, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 0),
            2
        )


        # ----------------------------------------------------
        # Eye state
        # ----------------------------------------------------

        cv2.putText(
            display_frame,
            f"Eyes: "
            f"{eye_data['combined_state']}",
            (20, 145),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.75,
            (0, 255, 255),
            2
        )


        # ----------------------------------------------------
        # Closure duration
        # ----------------------------------------------------

        cv2.putText(
            display_frame,
            f"Closure: "
            f"{eye_temporal_state['current_closure_duration']:.2f}s",
            (20, 185),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )


        # ----------------------------------------------------
        # Last event
        # ----------------------------------------------------

        cv2.putText(
            display_frame,
            f"Last Event: "
            f"{eye_temporal_state['last_closure_event']}",
            (20, 225),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )


        # ----------------------------------------------------
        # Longest closure
        # ----------------------------------------------------

        cv2.putText(
            display_frame,
            f"Longest Closure: "
            f"{eye_temporal_state['longest_closure']:.2f}s",
            (20, 265),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 255, 255),
            2
        )


    else:

        cv2.putText(
            display_frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.90,
            (0, 0, 255),
            2
        )


    # --------------------------------------------------------
    # Display
    # --------------------------------------------------------

    cv2.imshow(
        "NB5 - Integrated Eye Analysis",
        display_frame
    )


    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


cv2.destroyAllWindows()

camera.release()


print("=" * 60)
print("EYE ANALYSIS TEST COMPLETE")
print("=" * 60)

print(
    "Longest Closure:",
    f"{eye_temporal_state['longest_closure']:.2f}s"
)

print(
    "Last Event:",
    eye_temporal_state["last_closure_event"]
)

print("=" * 60)

LIVE EYE ANALYSIS TEST
Press Q to stop.
EYE ANALYSIS TEST COMPLETE
Longest Closure: 0.22s
Last Event: NORMAL BLINK


## Part 3 End State

At the end of Part 3, the notebook should have:

Camera
    ↓
MediaPipe
    ↓
Shared face landmarks
    ↓
Eye analysis
    ↓
EAR
    ↓
Open / Closed
    ↓
Closure duration
    ↓
Blink / Long blink / Prolonged closure

Do NOT add a drowsiness score in Part 3.

Do NOT add mouth prediction in Part 3.

Do NOT add eye + mouth fusion in Part 3.

The only purpose of this part is to verify that the eye-analysis
module works correctly inside the shared NB5 pipeline.

In [53]:
# ============================================================
# PART 4 — MOUTH CROP
# ============================================================

def crop_mouth_from_landmarks(
    frame,
    landmarks
):

    """
    Crop the mouth region using already-detected
    MediaPipe face landmarks.

    IMPORTANT:
    MediaPipe is NOT called inside this function.
    """

    if frame is None:
        return None


    if landmarks is None:
        return None


    h, w = frame.shape[:2]


    # --------------------------------------------------------
    # Collect mouth landmark coordinates
    # --------------------------------------------------------

    points = []


    for idx in MOUTH_POINTS:

        x = int(
            landmarks[idx].x * w
        )

        y = int(
            landmarks[idx].y * h
        )

        points.append(
            [x, y]
        )


    points = np.array(
        points,
        dtype=np.int32
    )


    # --------------------------------------------------------
    # Bounding rectangle
    # --------------------------------------------------------

    x, y, bw, bh = cv2.boundingRect(
        points
    )


    # --------------------------------------------------------
    # Same padding used during preprocessing
    # --------------------------------------------------------

    pad_x = int(
        bw * 0.30
    )

    pad_y = int(
        bh * 0.40
    )


    x1 = max(
        0,
        x - pad_x
    )

    y1 = max(
        0,
        y - pad_y
    )

    x2 = min(
        w,
        x + bw + pad_x
    )

    y2 = min(
        h,
        y + bh + pad_y
    )


    # --------------------------------------------------------
    # Crop
    # --------------------------------------------------------

    mouth = frame[
        y1:y2,
        x1:x2
    ]


    if mouth.size == 0:

        return None


    # --------------------------------------------------------
    # Resize to CNN input size
    # --------------------------------------------------------

    mouth = cv2.resize(
        mouth,
        (128, 128),
        interpolation=cv2.INTER_AREA
    )


    return mouth


print("Mouth crop function ready.")

Mouth crop function ready.


In [54]:
# ============================================================
# MOUTH CNN PREDICTION
# ============================================================

def predict_mouth_state(
    mouth_image
):

    """
    Predict the mouth state using the already-loaded
    trained CNN.

    Returns:
        prediction
        confidence
        probabilities
    """

    if mouth_image is None:

        return (
            None,
            0.0,
            None
        )


    # --------------------------------------------------------
    # OpenCV BGR -> RGB
    # --------------------------------------------------------

    mouth_rgb = cv2.cvtColor(
        mouth_image,
        cv2.COLOR_BGR2RGB
    )


    # --------------------------------------------------------
    # RGB -> PIL grayscale
    # --------------------------------------------------------

    mouth_pil = (
        Image
        .fromarray(mouth_rgb)
        .convert("L")
    )


    # --------------------------------------------------------
    # Same preprocessing used during training
    # --------------------------------------------------------

    input_tensor = mouth_transform(
        mouth_pil
    )


    input_tensor = (
        input_tensor
        .unsqueeze(0)
        .to(device)
    )


    # --------------------------------------------------------
    # CNN inference
    # --------------------------------------------------------

    with torch.no_grad():

        outputs = mouth_model(
            input_tensor
        )


        probabilities = torch.softmax(
            outputs,
            dim=1
        )


        predicted_index = torch.argmax(
            probabilities,
            dim=1
        ).item()


        confidence = (
            probabilities[
                0,
                predicted_index
            ].item()
            * 100.0
        )


    prediction = CLASS_NAMES[
        predicted_index
    ]


    return (
        prediction,
        confidence,
        probabilities[0]
        .detach()
        .cpu()
        .numpy()
    )


print("Mouth CNN prediction function ready.")

Mouth CNN prediction function ready.


In [55]:
# ============================================================
# MOUTH TEMPORAL STATE
# ============================================================

from collections import deque


MOUTH_HISTORY_SIZE = 20


mouth_prediction_history = deque(
    maxlen=MOUTH_HISTORY_SIZE
)


mouth_temporal_state = {

    "current_prediction": "None",

    "current_confidence": 0.0,

    "smoothed_prediction": "None",

    "smoothed_count": 0,

    "yawn_count": 0
}


print("=" * 60)
print("MOUTH TEMPORAL STATE INITIALIZED")
print("=" * 60)

print(
    "History size:",
    MOUTH_HISTORY_SIZE
)

print("=" * 60)

MOUTH TEMPORAL STATE INITIALIZED
History size: 20


In [56]:
# ============================================================
# UPDATE MOUTH TEMPORAL STATE
# ============================================================

def update_mouth_temporal_state(
    prediction,
    confidence,
    state
):

    """
    Update recent mouth predictions and calculate
    the smoothed mouth state.
    """

    if prediction is None:

        return state


    # --------------------------------------------------------
    # Add prediction to history
    # --------------------------------------------------------

    mouth_prediction_history.append(
        prediction
    )


    # --------------------------------------------------------
    # Current prediction
    # --------------------------------------------------------

    state[
        "current_prediction"
    ] = prediction


    state[
        "current_confidence"
    ] = confidence


    # --------------------------------------------------------
    # Most common recent prediction
    # --------------------------------------------------------

    counts = {}


    for item in mouth_prediction_history:

        counts[item] = (
            counts.get(item, 0)
            + 1
        )


    smoothed_prediction = max(
        counts,
        key=counts.get
    )


    state[
        "smoothed_prediction"
    ] = smoothed_prediction


    state[
        "smoothed_count"
    ] = counts[
        smoothed_prediction
    ]


    # --------------------------------------------------------
    # Recent Yawn count
    # --------------------------------------------------------

    state[
        "yawn_count"
    ] = sum(
        item == "Yawn"
        for item in mouth_prediction_history
    )


    return state


print("Mouth temporal update function ready.")

Mouth temporal update function ready.


In [58]:
# ============================================================
# PART 4 — LIVE EYE + MOUTH TEST
# ============================================================

print("=" * 60)
print("LIVE EYE + MOUTH ANALYSIS TEST")
print("=" * 60)
print("Press Q to stop.")
print("=" * 60)


# ------------------------------------------------------------
# Reopen shared camera if necessary
# ------------------------------------------------------------

if not camera.is_open:

    if not camera.open():

        raise RuntimeError(
            f"Could not open camera "
            f"index {camera.camera_index}"
        )


# ------------------------------------------------------------
# Reset eye temporal state
# ------------------------------------------------------------

eye_temporal_state = {

    "eyes_closed": False,

    "closure_start_time": None,

    "current_closure_duration": 0.0,

    "last_completed_closure": 0.0,

    "last_closure_event": "None",

    "longest_closure": 0.0
}


# ------------------------------------------------------------
# Reset mouth temporal state
# ------------------------------------------------------------

mouth_prediction_history.clear()


mouth_temporal_state = {

    "current_prediction": "None",

    "current_confidence": 0.0,

    "smoothed_prediction": "None",

    "smoothed_count": 0,

    "yawn_count": 0
}


while True:

    # ========================================================
    # GET ONE SHARED CAMERA FRAME
    # ========================================================

    ret, frame = camera.read()


    if not ret:

        print(
            "Failed to read camera frame."
        )

        break


    display_frame = frame.copy()


    # ========================================================
    # ONE SHARED MEDIAPIPE CALL
    # ========================================================

    landmarks = detect_face_landmarks(
        frame
    )


    if landmarks is not None:

        h, w = frame.shape[:2]


        # ====================================================
        # EYE ANALYSIS
        # ====================================================

        eye_data = analyze_eyes(
            landmarks,
            w,
            h
        )


        eye_temporal_state = (
            update_eye_temporal_state(
                eye_data["both_closed"],
                time.time(),
                eye_temporal_state
            )
        )


        # ====================================================
        # MOUTH CROP
        # ====================================================

        mouth = crop_mouth_from_landmarks(
            frame,
            landmarks
        )


        # ====================================================
        # MOUTH CNN
        # ====================================================

        mouth_prediction = (
            predict_mouth_state(
                mouth
            )
        )


        prediction = mouth_prediction[0]

        confidence = mouth_prediction[1]


        mouth_temporal_state = (
            update_mouth_temporal_state(
                prediction,
                confidence,
                mouth_temporal_state
            )
        )


        # ====================================================
        # DRAW EYE LANDMARKS
        # ====================================================

        for idx in LEFT_EYE:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                3,
                (0, 255, 0),
                -1
            )


        for idx in RIGHT_EYE:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                3,
                (0, 255, 0),
                -1
            )


        # ====================================================
        # DRAW MOUTH LANDMARKS
        # ====================================================

        for idx in MOUTH_POINTS:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                2,
                (255, 0, 0),
                -1
            )


        # ====================================================
        # DISPLAY EYE INFORMATION
        # ====================================================

        cv2.putText(
            display_frame,
            f"Left EAR: "
            f"{eye_data['left_ear']:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Right EAR: "
            f"{eye_data['right_ear']:.3f}",
            (20, 65),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Eyes: "
            f"{eye_data['combined_state']}",
            (20, 100),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 255),
            2
        )


        cv2.putText(
            display_frame,
            f"Eye Closure: "
            f"{eye_temporal_state['current_closure_duration']:.2f}s",
            (20, 135),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )


        # ====================================================
        # DISPLAY MOUTH INFORMATION
        # ====================================================

        cv2.putText(
            display_frame,
            f"Mouth: "
            f"{mouth_temporal_state['current_prediction']}",
            (20, 175),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 150, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Mouth Conf: "
            f"{mouth_temporal_state['current_confidence']:.1f}%",
            (20, 210),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 150, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Mouth Smoothed: "
            f"{mouth_temporal_state['smoothed_prediction']}",
            (20, 245),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Recent Yawns: "
            f"{mouth_temporal_state['yawn_count']}/"
            f"{len(mouth_prediction_history)}",
            (20, 280),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 0),
            2
        )


    else:

        cv2.putText(
            display_frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.90,
            (0, 0, 255),
            2
        )


    # ========================================================
    # DISPLAY CAMERA
    # ========================================================

    cv2.imshow(
        "NB5 - Integrated Eye + Mouth Test",
        display_frame
    )


    # ========================================================
    # DISPLAY MOUTH CROP
    # ========================================================

    if landmarks is not None:

        if mouth is not None:

            cv2.imshow(
                "NB5 - Mouth Crop",
                mouth
            )


    # ========================================================
    # QUIT
    # ========================================================

    if cv2.waitKey(1) & 0xFF == ord("q"):

        break


cv2.destroyAllWindows()

camera.release()


print("=" * 60)
print("PART 4 TEST COMPLETE")
print("=" * 60)

print(
    "Last mouth prediction:",
    mouth_temporal_state[
        "current_prediction"
    ]
)

print(
    "Smoothed mouth prediction:",
    mouth_temporal_state[
        "smoothed_prediction"
    ]
)

print(
    "Recent yawns:",
    mouth_temporal_state[
        "yawn_count"
    ]
)

print(
    "Longest eye closure:",
    f"{eye_temporal_state['longest_closure']:.2f}s"
)

print("=" * 60)

LIVE EYE + MOUTH ANALYSIS TEST
Press Q to stop.
PART 4 TEST COMPLETE
Last mouth prediction: Closed
Smoothed mouth prediction: Closed
Recent yawns: 0
Longest eye closure: 0.77s


## Part 4 End State

At the end of Part 4, the integrated notebook must have:

ONE camera
    ↓
ONE MediaPipe face-landmark detection
    ↓
shared landmarks
    ├── Eyes
    │    ↓
    │   EAR
    │    ↓
    │   eye state
    │    ↓
    │   closure duration
    │
    └── Mouth
         ↓
        crop
         ↓
       grayscale
         ↓
       128×128
         ↓
       CNN
         ↓
   Closed/Talking/Yawn

Do NOT add final drowsiness scoring yet.

Do NOT add eye + mouth fusion yet.

Do NOT add alerts yet.

Do NOT add a second camera.

Do NOT add a second MediaPipe detector.

The only purpose of Part 4 is to verify that eye and mouth
analysis can run simultaneously from the same camera frame and
same MediaPipe landmark result.

In [ ]:
# ============================================================
# PART 5 — DROWSINESS FUSION CONFIGURATION
# ============================================================

# ------------------------------------------------------------
# Signal weights
# ------------------------------------------------------------

EYE_WEIGHT = 0.70

MOUTH_WEIGHT = 0.30


# ------------------------------------------------------------
# Final drowsiness states
# ------------------------------------------------------------

NORMAL_THRESHOLD = 40.0

DROWSY_THRESHOLD = 70.0


# ------------------------------------------------------------
# Score decay
# ------------------------------------------------------------

FINAL_SCORE_DECAY_PER_SECOND = 1.5


# ------------------------------------------------------------
# Eye scoring parameters
# ------------------------------------------------------------

EYE_PROLONGED_POINTS = 30.0

EYE_LONG_BLINK_POINTS = 5.0


# ------------------------------------------------------------
# Mouth scoring parameters
# ------------------------------------------------------------

MOUTH_YAWN_POINTS = 20.0


# ------------------------------------------------------------
# History windows
# ------------------------------------------------------------

EYE_EVENT_HISTORY_SECONDS = 30

MOUTH_HISTORY_SIZE = 20


print("=" * 60)
print("DROWSINESS FUSION CONFIGURATION")
print("=" * 60)

print(
    "Eye weight   :",
    EYE_WEIGHT
)

print(
    "Mouth weight :",
    MOUTH_WEIGHT
)

print(
    "Normal       :",
    f"< {NORMAL_THRESHOLD}"
)

print(
    "Warning      :",
    f"{NORMAL_THRESHOLD} - "
    f"{DROWSY_THRESHOLD - 0.01:.2f}"
)

print(
    "Drowsy       :",
    f">= {DROWSY_THRESHOLD}"
)

print("=" * 60)

In [ ]:
# ============================================================
# EYE DROWSINESS SCORE
# ============================================================

def calculate_eye_score(
    eye_temporal_state,
    current_time,
    eye_event_history
):

    """
    Calculate an interpretable eye-based drowsiness score.

    Strong evidence:
        prolonged eye closure

    Supporting evidence:
        long blink

    Recent events are given more importance than old events.
    """

    score = 0.0


    # --------------------------------------------------------
    # Current prolonged closure
    # --------------------------------------------------------

    current_duration = (
        eye_temporal_state[
            "current_closure_duration"
        ]
    )


    if (
        eye_temporal_state[
            "eyes_closed"
        ]
        and
        current_duration
        >= PROLONGED_CLOSURE_THRESHOLD
    ):

        # Continuous score increase while prolonged closure
        # is still happening.

        extra_duration = min(
            current_duration,
            3.0
        )

        score += (
            EYE_PROLONGED_POINTS
            * (
                extra_duration
                / 3.0
            )
        )


    # --------------------------------------------------------
    # Recently completed closure events
    # --------------------------------------------------------

    recent_events = 0


    for event in eye_event_history:

        age = (
            current_time
            - event["time"]
        )


        if age <= EYE_EVENT_HISTORY_SECONDS:

            if (
                event["type"]
                == "PROLONGED CLOSURE"
            ):

                # Recent prolonged closures contribute strongly.

                recency_factor = max(
                    0.0,
                    1.0
                    - (
                        age
                        / EYE_EVENT_HISTORY_SECONDS
                    )
                )

                score += (
                    EYE_PROLONGED_POINTS
                    * recency_factor
                )

                recent_events += 1


            elif (
                event["type"]
                == "LONG BLINK"
            ):

                recency_factor = max(
                    0.0,
                    1.0
                    - (
                        age
                        / EYE_EVENT_HISTORY_SECONDS
                    )
                )

                score += (
                    EYE_LONG_BLINK_POINTS
                    * recency_factor
                )


    # --------------------------------------------------------
    # Clamp
    # --------------------------------------------------------

    score = min(
        100.0,
        max(
            0.0,
            score
        )
    )


    return score


print("Eye scoring function ready.")

In [ ]:
# ============================================================
# MOUTH DROWSINESS SCORE
# ============================================================

def calculate_mouth_score(
    mouth_prediction_history
):

    """
    Calculate a mouth-based supporting score.

    Yawn is treated as positive evidence.
    Talking and Closed are neutral.

    The more Yawn predictions appear in the recent history,
    the stronger the supporting mouth signal becomes.
    """

    if len(
        mouth_prediction_history
    ) == 0:

        return 0.0


    # --------------------------------------------------------
    # Count recent Yawn predictions
    # --------------------------------------------------------

    yawn_count = sum(
        prediction == "Yawn"
        for prediction
        in mouth_prediction_history
    )


    history_length = len(
        mouth_prediction_history
    )


    yawn_ratio = (
        yawn_count
        / history_length
    )


    # --------------------------------------------------------
    # Convert ratio to score
    # --------------------------------------------------------

    score = (
        yawn_ratio
        * 100.0
    )


    # --------------------------------------------------------
    # Apply supporting-signal limit
    # --------------------------------------------------------

    # Mouth alone should not dominate the system.

    score = min(
        100.0,
        max(
            0.0,
            score
        )
    )


    return score


print("Mouth scoring function ready.")

In [ ]:
# ============================================================
# FINAL DROWSINESS SCORE
# ============================================================

def calculate_final_drowsiness_score(
    eye_score,
    mouth_score
):

    """
    Combine eye and mouth signals using the configured weights.
    """

    final_score = (
        EYE_WEIGHT
        * eye_score
        +
        MOUTH_WEIGHT
        * mouth_score
    )


    final_score = min(
        100.0,
        max(
            0.0,
            final_score
        )
    )


    return final_score


def classify_drowsiness_state(
    final_score
):

    if final_score >= DROWSY_THRESHOLD:

        return "DROWSY"


    elif final_score >= NORMAL_THRESHOLD:

        return "WARNING"


    else:

        return "NORMAL"


print("Final fusion functions ready.")

In [ ]:
# ============================================================
# TEMPORAL FUSION STATE
# ============================================================

from collections import deque


eye_event_history = deque()


fusion_state = {

    "eye_score": 0.0,

    "mouth_score": 0.0,

    "final_score": 0.0,

    "state": "NORMAL",

    "last_update_time": time.time()

}


print("=" * 60)
print("TEMPORAL FUSION STATE INITIALIZED")
print("=" * 60)

print(fusion_state)

print("=" * 60)

In [ ]:
# ============================================================
# UPDATE FUSION STATE
# ============================================================

def update_fusion_state(
    eye_temporal_state,
    mouth_prediction_history,
    current_time,
    state,
    eye_events
):

    """
    Update eye score, mouth score and final drowsiness score.

    Also performs gradual score decay.

    """

    # --------------------------------------------------------
    # Add newly completed eye event
    # --------------------------------------------------------

    last_event = (
        eye_temporal_state[
            "last_closure_event"
        ]
    )


    last_completed_duration = (
        eye_temporal_state[
            "last_completed_closure"
        ]
    )


    if (
        last_event != "None"
        and
        last_completed_duration > 0
    ):

        already_recorded = False


        if len(eye_events) > 0:

            last_record = eye_events[-1]

            already_recorded = (
                abs(
                    last_record[
                        "duration"
                    ]
                    -
                    last_completed_duration
                )
                < 0.01
            )


        if not already_recorded:

            eye_events.append(
                {
                    "time": current_time,
                    "type": last_event,
                    "duration":
                        last_completed_duration
                }
            )


    # --------------------------------------------------------
    # Remove old eye events
    # --------------------------------------------------------

    cutoff = (
        current_time
        -
        EYE_EVENT_HISTORY_SECONDS
    )


    while (
        eye_events
        and
        eye_events[0]["time"]
        < cutoff
    ):

        eye_events.popleft()


    # --------------------------------------------------------
    # Calculate signals
    # --------------------------------------------------------

    eye_score = calculate_eye_score(
        eye_temporal_state,
        current_time,
        eye_events
    )


    mouth_score = calculate_mouth_score(
        mouth_prediction_history
    )


    # --------------------------------------------------------
    # Calculate final score
    # --------------------------------------------------------

    final_score = (
        calculate_final_drowsiness_score(
            eye_score,
            mouth_score
        )
    )


    # --------------------------------------------------------
    # Score decay / recovery
    # --------------------------------------------------------

    elapsed = (
        current_time
        -
        state["last_update_time"]
    )


    if (
        eye_score < NORMAL_THRESHOLD
        and
        mouth_score < NORMAL_THRESHOLD
    ):

        state["final_score"] -= (
            FINAL_SCORE_DECAY_PER_SECOND
            * elapsed
        )


        state["final_score"] = max(
            state["final_score"],
            0.0
        )


        # Allow current evidence to raise the score
        # again immediately.

        state["final_score"] = max(
            state["final_score"],
            final_score
        )

    else:

        state["final_score"] = final_score


    # --------------------------------------------------------
    # Clamp final score
    # --------------------------------------------------------

    state["final_score"] = min(
        100.0,
        max(
            0.0,
            state["final_score"]
        )
    )


    # --------------------------------------------------------
    # Update component scores
    # --------------------------------------------------------

    state["eye_score"] = eye_score

    state["mouth_score"] = mouth_score


    # --------------------------------------------------------
    # State classification
    # --------------------------------------------------------

    state["state"] = (
        classify_drowsiness_state(
            state["final_score"]
        )
    )


    state["last_update_time"] = (
        current_time
    )


    return state


print("Fusion update function ready.")

In [ ]:
# ============================================================
# PART 5 — LIVE DROWSINESS FUSION TEST
# ============================================================

print("=" * 60)
print("LIVE DROWSINESS FUSION TEST")
print("=" * 60)
print("Press Q to stop.")
print("=" * 60)


# ------------------------------------------------------------
# Reopen camera if necessary
# ------------------------------------------------------------

if not camera.is_open:

    if not camera.open():

        raise RuntimeError(
            f"Could not open camera "
            f"index {camera.camera_index}"
        )


# ------------------------------------------------------------
# Reset eye state
# ------------------------------------------------------------

eye_temporal_state = {

    "eyes_closed": False,

    "closure_start_time": None,

    "current_closure_duration": 0.0,

    "last_completed_closure": 0.0,

    "last_closure_event": "None",

    "longest_closure": 0.0
}


# ------------------------------------------------------------
# Reset mouth state
# ------------------------------------------------------------

mouth_prediction_history.clear()

mouth_temporal_state = {

    "current_prediction": "None",

    "current_confidence": 0.0,

    "smoothed_prediction": "None",

    "smoothed_count": 0,

    "yawn_count": 0
}


# ------------------------------------------------------------
# Reset eye events
# ------------------------------------------------------------

eye_event_history.clear()


# ------------------------------------------------------------
# Reset fusion state
# ------------------------------------------------------------

fusion_state = {

    "eye_score": 0.0,

    "mouth_score": 0.0,

    "final_score": 0.0,

    "state": "NORMAL",

    "last_update_time": time.time()

}


while True:

    # ========================================================
    # GET FRAME
    # ========================================================

    ret, frame = camera.read()


    if not ret:

        print(
            "Failed to read camera frame."
        )

        break


    display_frame = frame.copy()


    # ========================================================
    # ONE MEDIAPIPE CALL
    # ========================================================

    landmarks = detect_face_landmarks(
        frame
    )


    if landmarks is not None:

        h, w = frame.shape[:2]


        # ====================================================
        # EYE ANALYSIS
        # ====================================================

        eye_data = analyze_eyes(
            landmarks,
            w,
            h
        )


        eye_temporal_state = (
            update_eye_temporal_state(
                eye_data["both_closed"],
                time.time(),
                eye_temporal_state
            )
        )


        # ====================================================
        # MOUTH ANALYSIS
        # ====================================================

        mouth = crop_mouth_from_landmarks(
            frame,
            landmarks
        )


        prediction, confidence, _ = (
            predict_mouth_state(
                mouth
            )
        )


        if prediction is not None:

            mouth_temporal_state = (
                update_mouth_temporal_state(
                    prediction,
                    confidence,
                    mouth_temporal_state
                )
            )


        # ====================================================
        # UPDATE FUSION
        # ====================================================

        fusion_state = (
            update_fusion_state(
                eye_temporal_state,
                mouth_prediction_history,
                time.time(),
                fusion_state,
                eye_event_history
            )
        )


        # ====================================================
        # DRAW EYE LANDMARKS
        # ====================================================

        for idx in LEFT_EYE:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                3,
                (0, 255, 0),
                -1
            )


        for idx in RIGHT_EYE:

            x = int(
                landmarks[idx].x * w
            )

            y = int(
                landmarks[idx].y * h
            )

            cv2.circle(
                display_frame,
                (x, y),
                3,
                (0, 255, 0),
                -1
            )


        # ====================================================
        # DISPLAY EYE DATA
        # ====================================================

        cv2.putText(
            display_frame,
            f"EAR: "
            f"{eye_data['average_ear']:.3f}",
            (20, 35),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (0, 255, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Eyes: "
            f"{eye_data['combined_state']}",
            (20, 70),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 255),
            2
        )


        cv2.putText(
            display_frame,
            f"Eye Closure: "
            f"{eye_temporal_state['current_closure_duration']:.2f}s",
            (20, 105),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 255),
            2
        )


        cv2.putText(
            display_frame,
            f"Eye Score: "
            f"{fusion_state['eye_score']:.1f}",
            (20, 140),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (0, 255, 255),
            2
        )


        # ====================================================
        # DISPLAY MOUTH DATA
        # ====================================================

        cv2.putText(
            display_frame,
            f"Mouth: "
            f"{mouth_temporal_state['current_prediction']}",
            (20, 180),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 150, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Mouth Score: "
            f"{fusion_state['mouth_score']:.1f}",
            (20, 215),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.70,
            (255, 150, 0),
            2
        )


        cv2.putText(
            display_frame,
            f"Recent Yawns: "
            f"{mouth_temporal_state['yawn_count']}/"
            f"{len(mouth_prediction_history)}",
            (20, 250),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.65,
            (255, 255, 0),
            2
        )


        # ====================================================
        # DISPLAY FINAL SCORE
        # ====================================================

        cv2.putText(
            display_frame,
            f"FINAL DROWSINESS SCORE: "
            f"{fusion_state['final_score']:.1f}/100",
            (20, 300),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.80,
            (0, 255, 255),
            2
        )


        # ====================================================
        # DISPLAY FINAL STATE
        # ====================================================

        final_state = fusion_state[
            "state"
        ]


        if final_state == "DROWSY":

            state_color = (
                0,
                0,
                255
            )

        elif final_state == "WARNING":

            state_color = (
                0,
                165,
                255
            )

        else:

            state_color = (
                0,
                255,
                0
            )


        cv2.putText(
            display_frame,
            f"STATUS: {final_state}",
            (20, 345),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.90,
            state_color,
            3
        )


    else:

        cv2.putText(
            display_frame,
            "FACE NOT DETECTED",
            (20, 45),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.90,
            (0, 0, 255),
            2
        )


    # ========================================================
    # DISPLAY
    # ========================================================

    cv2.imshow(
        "NB5 - Integrated Drowsiness System",
        display_frame
    )


    if (
        cv2.waitKey(1)
        & 0xFF
        == ord("q")
    ):

        break


cv2.destroyAllWindows()

camera.release()


print("=" * 60)
print("PART 5 FUSION TEST COMPLETE")
print("=" * 60)

print(
    "Final Score:",
    f"{fusion_state['final_score']:.1f}/100"
)

print(
    "Final State:",
    fusion_state["state"]
)

print(
    "Eye Score:",
    f"{fusion_state['eye_score']:.1f}"
)

print(
    "Mouth Score:",
    f"{fusion_state['mouth_score']:.1f}"
)

print("=" * 60)

## Part 5 End State

The integrated system should now perform:

Camera
    ↓
One MediaPipe face-landmark detection
    ↓
    ┌─────────────────────────────┐
    │                             │
    ↓                             ↓
  EYES                           MOUTH
    ↓                             ↓
  EAR                            CNN
    ↓                             ↓
Eye state                      Yawn state
    ↓                             ↓
Closure duration              Yawn history
    ↓                             ↓
Eye Score                     Mouth Score
    └──────────────┬──────────────┘
                   ↓
             Weighted Fusion
                   ↓
        Final Drowsiness Score
                   ↓
       NORMAL / WARNING / DROWSY

IMPORTANT:
- This is a prototype scoring system.
- Eye weight = 70%.
- Mouth weight = 30%.
- Do not claim the score is clinically validated.
- Do not add an alert yet.
- Do not add external hardware yet.